# 06 · Gradient boosting (LightGBM, sklearn HistGradientBoosting)

Trees built one after another, each fixing the previous ones' errors — usually the most accurate family on
tabular data, but the easiest to overfit on ~1,000 rows. GB-04 adds **monotone constraints**: more past payments can
never lower the score; a larger balance, older debt or a longer time since the last payment can never raise it.
That keeps the model explainable and safer when real data looks different from training data.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "06"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
LGB_GRID = {"lgbmclassifier__num_leaves": randint(3, 32), "lgbmclassifier__learning_rate": loguniform(.01, .2),
            "lgbmclassifier__n_estimators": randint(50, 500), "lgbmclassifier__min_child_samples": randint(10, 80),
            "lgbmclassifier__reg_lambda": loguniform(1e-3, 10), "lgbmclassifier__subsample": uniform(.6, .4),
            "lgbmclassifier__subsample_freq": [1], "lgbmclassifier__colsample_bytree": uniform(.5, .5)}
run("GB-01", M.pipeline(M.preprocess("none"), HistGradientBoostingClassifier(random_state=M.SEED)), "FS_ALL", "sklearn HistGradientBoosting, defaults")
run("GB-02", M.pipeline(M.preprocess("none"), M.lgbm()), "FS_ALL", "LightGBM, defaults")
run("GB-03", M.tuned(M.pipeline(M.preprocess("none"), M.lgbm()), LGB_GRID, n_iter=20), "FS_ALL", "LightGBM, tuned (leaves, rate, trees, min child, L2, sampling)")
run("GB-04", M.tuned(M.pipeline(M.preprocess("none"), M.lgbm(M.FS_ALL, monotone=True)), LGB_GRID, n_iter=20), "FS_ALL", "LightGBM tuned + monotone constraints")
from sklearn.feature_selection import SelectFromModel
run("GB-05", M.tuned(M.pipeline(M.preprocess("none"), M.TopKPermutation(k=6), M.lgbm()), LGB_GRID, n_iter=20), "FS_ALL", "Top-6 features by permutation importance, then tuned LightGBM")
run("GB-06", M.tuned(M.pipeline(M.preprocess("none"), M.lgbm(M.FS_TRANSFER, monotone=True)), LGB_GRID, n_iter=20), "FS_TRANSFER", "LightGBM tuned + monotone, FS_TRANSFER");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

## SHAP: how the monotone, transferable model (GB-06) uses each feature

In [ ]:
import shap
X, y = M.xy(dev, M.FS_TRANSFER)
search = M.tuned(M.pipeline(M.preprocess("none"), M.lgbm(M.FS_TRANSFER, monotone=True)), LGB_GRID, n_iter=20).fit(X, y)
best = search.best_estimator_; Xf = best[:-1].transform(X)
sv = shap.TreeExplainer(best[-1]).shap_values(Xf)
sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, Xf, show=False, max_display=12); plt.title("SHAP values (positive = more likely to pay)"); plt.show()
print("best parameters:", {k.split("__")[1]: (round(v, 4) if isinstance(v, float) else v) for k, v in search.best_params_.items()})

## Findings (Salud-tuned world)
- Tuned LightGBM (GB-03) reaches **0.883**. Defaults reach 0.879 and are badly calibrated (slope 0.53).
  The notebook took 39 minutes, mostly tuning, for +0.004 (p = 0.53).
- Monotone constraints (GB-04, 0.877) cost 0.006 (p = 0.13). On FS_TRANSFER (GB-06, 0.876) the model is under-confident (slope 1.92), so it needs calibration (see notebook 11).
- Six features chosen by permutation importance (GB-05, 0.881) do as well as all 25.